# Session 3: Visualizing Dependencies and Input Requirements

This notebook uses a minimal custom input and DAG plots to understand which variables are needed to compute `einkommensteuer__betrag_y_sn` and `solidaritätszuschlag__betrag_y_sn`.

Use it as a live-coding notebook: run a step, inspect output, and discuss what the DAG and error messages imply.

## 0) Setup

Import core GETTSIM objects and set the policy date used throughout the notebook.

In [ ]:
# Command for installing gettsim (only use on Colab)

# !apt-get update -qq
# !apt-get install -y graphviz graphviz-dev pkg-config
# !pip install pygraphviz
# !pip install gettsim
# !pip install git+https://github.com/ttsim-dev/gettsim-personas.git

In [1]:
from gettsim import plot, main

In [2]:
POLICY_DATE = "2025-01-01"

## 1) Define Target Set

We intentionally request only two outputs to keep diagnostics interpretable.

In [3]:
from gettsim import TTTargets

# Small target set on purpose: this keeps diagnostics interpretable.
# Each `None` is a leaf marker ("request this target"), not missing data.

targets_small = {
    "einkommensteuer": {"betrag_y_sn": None},
    "solidaritätszuschlag": {"betrag_y_sn": None},
}

tt_targets_small = TTTargets(tree=targets_small)

### Step 2: Input Data

Create a minimal person-level DataFrame. One column uses a custom name to demonstrate mapping.

In [4]:
import pandas as pd

# Minimal toy data for learning diagnostics.
# This table is intentionally incomplete for full simulation runs.
# `capital_income_yearly` is a custom name and will be mapped later.

df_example = pd.DataFrame(
    {
        "p_id": [0, 1, 2],
        "hh_id": [0, 0, 0],
        "alter": [35, 34, 8],
        "arbeitsstunden_w": [40.0, 30.0, 0.0],
        "bruttolohn_m": [4200.0, 2400.0, 0.0],
        "capital_income_yearly": [120.0, 0.0, 0.0],
    }
)

df_example

,p_id,hh_id,alter,arbeitsstunden_w,bruttolohn_m,capital_income_yearly
0,0,0,35,40.0,4200.0,120.0
1,1,0,34,30.0,2400.0,0.0
2,2,0,8,0.0,0.0,0.0


### Step 3: Mapper

Map raw DataFrame columns into GETTSIM namespace paths. Then build `InputData.df_and_mapper(...)`.

In [5]:
from gettsim import InputData

mapper_example = {
    "p_id": "p_id",
    "hh_id": "hh_id",
    "alter": "alter",
    "arbeitsstunden_w": "arbeitsstunden_w",
    "einnahmen": {
        "bruttolohn_m": "bruttolohn_m",
        "kapitalerträge_y": "capital_income_yearly",
    },
}

input_data_df_mapper = InputData.df_and_mapper(df=df_example, mapper=mapper_example)


## 2) Diagnostic Run

Run the model with intentionally incomplete inputs and inspect the first missing root. This guides which branch to add next.

### Step 4: First Missing Input

Execute `main(...)` and inspect the first missing requirement. Use this as the next branch to add during live coding.

In [6]:
from gettsim import MainTarget

# Intentional diagnostic run.
# We expect this to fail and use the error message as a requirements list.
# This is the core workflow for building inputs target-by-target.

try:
    main(
        main_target=MainTarget.results.tree,
        policy_date_str="2025-01-01",
        input_data=input_data_df_mapper,
        tt_targets=tt_targets_small,
    )
except Exception as err:
    print(err)

The following data columns are missing.

[
    "('sozialversicherung', 'rente', 'bezieht_rente')",
    "('familie', 'p_id_ehepartner')",
    "('familie', 'p_id_elternteil_1')",
    "('familie', 'p_id_elternteil_2')",
    "('einkommensteuer', 'gemeinsam_veranlagt')",
    "('behinderungsgrad',)",
    "('familie', 'alleinerziehend')",
    "('einkommensteuer', 'einkünfte', 'aus_forst_und_landwirtschaft', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_gewerbebetrieb', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_selbstständiger_arbeit', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'aus_vermietung_und_verpachtung', 'betrag_y')",
    "('einkommensteuer', 'einkünfte', 'sonstige', 'alle_weiteren_y')",
    "('sozialversicherung', 'rente', 'jahr_renteneintritt')",
    "('einnahmen', 'renten', 'geförderte_private_vorsorge_m')",
    "('einnahmen', 'renten', 'sonstige_private_vorsorge_m')",
    "('einnahmen', 'renten', 'betriebliche_altersvorsorge_m')",
    "('einnahmen'

## 3) Visualize the Relevant DAG

Plot ancestors of the two requested targets. This reveals the computation neighborhood we need to satisfy.

### DAG Preview

This plot shows ancestors of the selected targets and helps explain why specific inputs are required.

### Selection Types in `plot.dag.tt(...)`

Useful `selection_type` options:
- `"ancestors"`: upstream dependencies of selected nodes.
- `"descendants"`: downstream effects of selected nodes.
- `"neighbors"`: direct one-step neighborhood.
- `"all_paths"`: nodes on paths between selected sets.

For input-requirement analysis, `"ancestors"` is usually the most informative choice.

In [7]:
plot.dag.tt(
    policy_date_str=POLICY_DATE,
    primary_nodes={
        ("einkommensteuer", "betrag_y_sn"),
        ("solidaritätszuschlag", "betrag_y_sn"),
    },
    selection_type="all_paths",
    selection_depth=3,
    width=900,
    height=520,
)